# NN_11 — CNN Feature Extractor + SVM (Abordagem Híbrida)

Compara dois fluxos de extração de features + SVM:

| Fluxo | CNN | GAP dim | Input CNN |
|-------|-----|---------|-----------|
| **1D CNN-SVM** (original) | 1D CNN z_eq (NN_02 GPU) | 128-d | z_eq ∈ R¹⁰²⁴ |
| **4-feat CNN-SVM** (novo) | CNN 4-feat (NN_14) | 32-d | [|τ|, ĥ, SNR, E] |

**Hipótese**: A CNN que processa z_eq aprende representações mais ricas do sinal
do que a CNN sobre 4 features escalares — logo, o SVM treinado sobre features de 128d
deve superar o SVM sobre features de 32d.

Threshold via D3F Gaussiano (Braca 2022): τ*(α) = μ_H0 + Q⁻¹(1−α)·σ_H0 com α=10⁻⁷.


In [ ]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import numpy as np
import matplotlib.pyplot as plt
import h5py
import json
from pathlib import Path
from scipy.stats import norm
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import gc
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras

print(f'TensorFlow : {tf.__version__}')
print(f'GPUs       : {tf.config.list_physical_devices("GPU")}')

notebook_dir = Path.cwd()
project_root = notebook_dir.parent
results_dir  = project_root / 'results'
data_dir     = results_dir / 'data'
models_dir   = results_dir / 'models'
vis_dir      = results_dir / 'visualizations'

ALPHA    = 1e-7
Q_INV    = norm.ppf(1 - ALPHA)
SNR_BINS = [0, 5, 10, 15, 20, 25, 30]
print(f'\nα={ALPHA:.0e}  Q⁻¹={Q_INV:.4f}')

In [ ]:
# ==============================================================================
# 2. CARREGAR DATASET (apenas labels/SNR — sinais carregados em chunks na cell 4)
# ==============================================================================
dataset_path = data_dir / 'dataset_cnn_yeq_0_30dB.h5'

with h5py.File(str(dataset_path), 'r') as f:
    y_train   = f['train/y'][:].astype(np.float32)
    snr_train = f['train/snr'][:]
    y_val     = f['val/y'][:].astype(np.float32)
    snr_val   = f['val/snr'][:]
    y_test    = f['test/y'][:].astype(np.float32)
    snr_test  = f['test/snr'][:]
    L_FIXED   = int(f.attrs['L_FIXED'])

print(f'train={len(y_train)}  val={len(y_val)}  test={len(y_test)}  L={L_FIXED}')

# Carregar referência
with open(str(data_dir / 'nn08_architecture_comparison.json')) as f:
    ref = json.load(f)
classical_pd  = {int(k): v for k, v in ref['classical'].items()}
cnn_sigmoid_pd = {int(k): v for k, v in ref.get('cnn_sigmoid', {}).items()}
if not cnn_sigmoid_pd:
    with open(str(data_dir / 'nn09_logit_d3f.json')) as f:
        r09 = json.load(f)
    cnn_sigmoid_pd = {int(k): v for k, v in r09['cnn_sigmoid'].items()}
print('Referências carregadas.')


In [ ]:
# ==============================================================================
# 3. CARREGAR CNN E CRIAR EXTRATORES
# ==============================================================================
model_cnn = keras.models.load_model(str(models_dir / 'cnn1d_tag_auth_best.keras'))
print('1D CNN carregada.')

# Extrator de features 128-dim (saída do GlobalAveragePooling1D)
feature_extractor = keras.Model(
    inputs=model_cnn.input,
    outputs=model_cnn.get_layer('global_average_pooling1d').output,
    name='cnn_feature_extractor'
)
print(f'Feature extractor: saída = {feature_extractor.output.shape}')

# Extrator de logit escalar: Dense(1) sem sigmoid com os pesos de P_H1
_p_layer = model_cnn.get_layer('P_H1')
_kernel, _bias = _p_layer.get_weights()
_logit_dense = keras.layers.Dense(1, activation=None, use_bias=True, name='logit_linear')(_p_layer.input)
logit_extractor = keras.Model(inputs=model_cnn.input, outputs=_logit_dense, name='cnn_logit_extractor')
logit_extractor.get_layer('logit_linear').set_weights([_kernel, _bias])
print(f'Logit extractor: saída = {logit_extractor.output.shape}')


In [ ]:
# ==============================================================================
# 4. EXTRAIR FEATURES COM GPU (em chunks do h5 para economizar RAM)
# ==============================================================================
BATCH = 512

@tf.function(reduce_retracing=True)
def extract_batch(x):
    return feature_extractor(x, training=False), logit_extractor(x, training=False)

def extract_all(split, n):
    feats, logits = [], []
    with h5py.File(str(dataset_path), 'r') as hf:
        for start in range(0, n, BATCH):
            chunk = hf[f'{split}/y_eq'][start:start+BATCH].astype(np.float32)
            chunk = chunk.reshape(-1, L_FIXED, 1)
            f_out, z_out = extract_batch(chunk)
            feats.append(f_out.numpy())
            logits.append(z_out.numpy())
    return np.concatenate(feats), np.concatenate(logits).flatten()

print('Extraindo train...')
F_train, z_train = extract_all('train', len(y_train))
print('Extraindo val...')
F_val,   z_val   = extract_all('val',   len(y_val))
print('Extraindo test...')
F_test,  z_test  = extract_all('test',  len(y_test))

print(f'Features: train={F_train.shape}  val={F_val.shape}  test={F_test.shape}')
print(f'AUC logit (referência): {roc_auc_score(y_test, z_test):.5f}')


In [ ]:
# ==============================================================================
# 5. TREINAR SVM NAS FEATURES CNN
# ==============================================================================
print('Treinando SVM linear nas features CNN (128-dim)...')
svm_cnn_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SGDClassifier(loss='hinge', alpha=1e-4, max_iter=200,
                             tol=1e-4, random_state=42, n_jobs=-1,
                             class_weight='balanced'))
])
svm_cnn_linear.fit(F_train, y_train)
auc_lin = roc_auc_score(y_val, svm_cnn_linear.decision_function(F_val))
print(f'  SVM-Linear val AUC = {auc_lin:.5f}')

print('\nTreinando SVM com kernel RBF aproximado (Nystroem 300 comp)...')
svm_cnn_rbf = Pipeline([
    ('scaler',   StandardScaler()),
    ('nystroem', Nystroem(kernel='rbf', gamma=0.05, n_components=300, random_state=42)),
    ('svm',      SGDClassifier(loss='hinge', alpha=1e-4, max_iter=200,
                               tol=1e-4, random_state=42, n_jobs=-1,
                               class_weight='balanced'))
])
svm_cnn_rbf.fit(F_train, y_train)
auc_rbf = roc_auc_score(y_val, svm_cnn_rbf.decision_function(F_val))
print(f'  SVM-RBF  val AUC = {auc_rbf:.5f}')

best_svm_cnn = svm_cnn_rbf if auc_rbf >= auc_lin else svm_cnn_linear
best_name    = 'CNN-SVM-RBF' if auc_rbf >= auc_lin else 'CNN-SVM-Linear'
auc_best     = max(auc_lin, auc_rbf)
print(f'\nMelhor: {best_name}  val AUC={auc_best:.5f}')

In [ ]:
# ==============================================================================
# 6. D3F SOBRE TODOS OS ESCORES
# ==============================================================================
def d3f_eval(scores, y_true, snr, alpha=1e-7, clip=None):
    results = {}; thresholds = {}
    for snr_db in SNR_BINS:
        m0 = (snr >= snr_db-2.5) & (snr < snr_db+2.5) & (y_true==0)
        m1 = (snr >= snr_db-2.5) & (snr < snr_db+2.5) & (y_true==1)
        if m0.sum() < 30 or m1.sum() < 10:
            results[snr_db]=None; thresholds[snr_db]=None; continue
        mu, sigma = scores[m0].mean(), scores[m0].std()
        thr = float(mu + norm.ppf(1-alpha)*sigma)
        if clip: thr = np.clip(thr, clip[0], clip[1])
        results[snr_db]    = float((scores[m1] > thr).mean())
        thresholds[snr_db] = thr
    return results, thresholds

# Escores CNN-SVM
scores_svm = best_svm_cnn.decision_function(F_test)

pd_cnn_svm, thr_cnn_svm = d3f_eval(scores_svm, y_test, snr_test)
pd_cnn_log, thr_cnn_log = d3f_eval(z_test,     y_test, snr_test)

print('SNR  | Classical | CNN-Sigmoid | CNN-Logit | CNN-SVM')
print('-' * 60)
for snr_db in SNR_BINS:
    cl  = classical_pd.get(snr_db, 0)
    cs  = cnn_sigmoid_pd.get(snr_db, 0)
    cl2 = pd_cnn_log.get(snr_db) or 0
    sv  = pd_cnn_svm.get(snr_db) or 0
    print(f'{snr_db:3d}  | {cl:8.4f}  | {cs:10.4f}  | {cl2:8.4f}  | {sv:7.4f}')

In [ ]:
# ==============================================================================
# 7. VISUALIZAÇÃO
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CNN Feature Extractor + SVM vs métodos base (α=10⁻⁷)', fontsize=12)

snr_v = SNR_BINS
g = lambda d, k: [d.get(s) or 0 for s in k]

ax = axes[0]
ax.plot(snr_v, g(classical_pd,    snr_v), 'k-o',  lw=2, label='Clássico')
ax.plot(snr_v, g(cnn_sigmoid_pd,  snr_v), 'b--^', lw=1.5, alpha=0.7, label='CNN-Sigmoid')
ax.plot(snr_v, g(pd_cnn_log,      snr_v), 'b-^',  lw=2, label='CNN-Logit')
ax.plot(snr_v, g(pd_cnn_svm,      snr_v), 'g-s',  lw=2, label=best_name)
ax.set(xlabel='SNR (dB)', ylabel='PD', title='PD vs SNR', ylim=(-0.05,1.05))
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
# Scatter: feature espaço 2D (PCA)
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(F_test)
F2  = pca.transform(F_test)
for snr_db, color in [(5,'royalblue'), (20,'tomato'), (30,'seagreen')]:
    m = (snr_test >= snr_db-2.5) & (snr_test < snr_db+2.5)
    ax.scatter(F2[m & (y_test==0), 0], F2[m & (y_test==0), 1],
               alpha=0.2, s=2, color=color, marker='x')
    ax.scatter(F2[m & (y_test==1), 0], F2[m & (y_test==1), 1],
               alpha=0.2, s=2, color=color, marker='o')
ax.set(xlabel='PC1', ylabel='PC2', title='CNN Features PCA (×=H₀, o=H₁)')
ax.grid(alpha=0.3)

ax = axes[2]
# Distribuição dos escores SVM por SNR
for snr_db, color in [(5,'royalblue'), (10,'orange'), (20,'tomato')]:
    m0 = (snr_test >= snr_db-2.5) & (snr_test < snr_db+2.5) & (y_test==0)
    if m0.sum() > 30:
        ax.hist(scores_svm[m0], bins=40, density=True, alpha=0.4,
                color=color, label=f'H₀ {snr_db}dB')
ax.set(xlabel='SVM decision score', ylabel='Densidade',
       title='Distribuição H₀ — CNN-SVM')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
vis_path = vis_dir / 'NN11_CNN_SVM_Hybrid.png'
plt.savefig(str(vis_path), dpi=120)
plt.show()
print(f'Salvo → {vis_path}')

In [ ]:
# ==============================================================================
# 8. SALVAR RESULTADOS
# ==============================================================================
import joblib
joblib.dump(best_svm_cnn, str(models_dir / 'svm_cnn_hybrid.pkl'))

results = {
    'alpha':          ALPHA,
    'method':         best_name,
    'val_auc':        float(auc_best),
    'cnn_features':   128,
    'classical':      {str(k): v for k, v in classical_pd.items()},
    'cnn_sigmoid':    {str(k): v for k, v in cnn_sigmoid_pd.items()},
    'cnn_logit_d3f':  {str(k): pd_cnn_log.get(k) for k in SNR_BINS},
    'cnn_svm_d3f':    {str(k): pd_cnn_svm.get(k) for k in SNR_BINS},
    'thresholds': {
        'cnn_logit': {str(k): thr_cnn_log.get(k) for k in SNR_BINS},
        'cnn_svm':   {str(k): thr_cnn_svm.get(k) for k in SNR_BINS},
    }
}

out_path = data_dir / 'nn11_cnn_svm_hybrid.json'
with open(str(out_path), 'w') as f:
    json.dump(results, f, indent=2)
print(f'Salvo → {out_path}')
print(f'\nResumo final:')
for snr_db in [5, 10, 15, 20, 25, 30]:
    sv = pd_cnn_svm.get(snr_db) or 0
    cl2 = pd_cnn_log.get(snr_db) or 0
    cl = classical_pd.get(snr_db, 0)
    delta = sv - cl2
    print(f'  {snr_db:2d}dB  CNN-SVM={sv:.3f}  CNN-Logit={cl2:.3f}  Δ={delta:+.3f}  Clássico={cl:.3f}')

## Parte 2 — CNN-SVM com Features da CNN 4-feat (NN_14)

In [ ]:
# ==============================================================================
# 9. CARREGAR FEATURES GAP DA CNN 4-feat (NN_14)
# ==============================================================================
# NN_14 treinou uma CNN sobre (4,1) e extraiu features GAP de 32 dimensões.
# Aqui carregamos essas features para treinar o SVM comparativamente.

gap_path = data_dir / 'cnn4feat_gap_features.h5'

if not gap_path.exists():
    raise FileNotFoundError(
        f"{gap_path} não encontrado.\nExecute NN_14_CNN_4feat.ipynb primeiro."
    )

with h5py.File(str(gap_path), 'r') as hf:
    GAP4_train = hf['train/gap'][:].astype(np.float32)
    GAP4_val   = hf['val/gap'][:].astype(np.float32)
    GAP4_test  = hf['test/gap'][:].astype(np.float32)
    GAP_DIM_4  = int(hf.attrs['gap_dim'])

print(f'CNN 4-feat GAP features: dim={GAP_DIM_4}')
print(f'Train: {GAP4_train.shape}  Val: {GAP4_val.shape}  Test: {GAP4_test.shape}')
print(f'Estatísticas: mean={GAP4_train.mean():.4f}  std={GAP4_train.std():.4f}')


In [ ]:
# ==============================================================================
# 10. TREINAR SVM NAS FEATURES CNN 4-feat (32-dim)
# ==============================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import Nystroem

print('SVM Linear nas features CNN 4-feat (32-dim)...')
svm4_lin = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SGDClassifier(loss='hinge', alpha=1e-4, max_iter=200,
                             tol=1e-4, random_state=42, n_jobs=-1,
                             class_weight='balanced'))
])
svm4_lin.fit(GAP4_train, y_train)
auc4_lin = roc_auc_score(y_val, svm4_lin.decision_function(GAP4_val))
print(f'  Linear val AUC = {auc4_lin:.5f}')

print('\nSVM RBF-Nystroem nas features CNN 4-feat (32-dim)...')
svm4_rbf = Pipeline([
    ('scaler',   StandardScaler()),
    ('nystroem', Nystroem(kernel='rbf', gamma=0.05, n_components=300, random_state=42)),
    ('svm',      SGDClassifier(loss='hinge', alpha=1e-4, max_iter=200,
                               tol=1e-4, random_state=42, n_jobs=-1,
                               class_weight='balanced'))
])
svm4_rbf.fit(GAP4_train, y_train)
auc4_rbf = roc_auc_score(y_val, svm4_rbf.decision_function(GAP4_val))
print(f'  RBF val AUC = {auc4_rbf:.5f}')

best_svm4 = svm4_rbf if auc4_rbf >= auc4_lin else svm4_lin
best_svm4_name = 'CNN4-SVM-RBF' if auc4_rbf >= auc4_lin else 'CNN4-SVM-Linear'
print(f'\nMelhor SVM 4-feat CNN: {best_svm4_name}')


In [ ]:
# ==============================================================================
# 11. D3F + COMPARAÇÃO FINAL: CNN-SVM 1D vs CNN-SVM 4-feat
# ==============================================================================

def d3f_pd_vs_snr(scores_val, scores_test, y_v, y_t, snr_v, snr_t,
                  snr_bins, alpha=1e-7, clip=None):
    q_inv = norm.ppf(1 - alpha)
    pd_dict, thr_dict = {}, {}
    for snr_db in snr_bins:
        m0 = (snr_v >= snr_db-2.5) & (snr_v < snr_db+2.5) & (y_v == 0)
        m1 = (snr_t >= snr_db-2.5) & (snr_t < snr_db+2.5) & (y_t == 1)
        if m0.sum() < 30 or m1.sum() < 5:
            pd_dict[snr_db] = None; thr_dict[snr_db] = None; continue
        mu, sig = scores_val[m0].mean(), scores_val[m0].std()
        thr = float(mu + q_inv * sig)
        if clip: thr = np.clip(thr, *clip)
        pd_dict[snr_db]  = float((scores_test[m1] > thr).mean())
        thr_dict[snr_db] = thr
    return pd_dict, thr_dict

# D3F para CNN-SVM 4-feat
sc4_val  = best_svm4.decision_function(GAP4_val).astype(np.float64)
sc4_test = best_svm4.decision_function(GAP4_test).astype(np.float64)
pd_cnn4_svm, _ = d3f_pd_vs_snr(sc4_val, sc4_test, y_val, y_test, snr_val, snr_test, SNR_BINS)

# D3F para melhor SVM da CNN 1D (já calculado acima como best_svm_cnn)
# Re-computar para garantir alinhamento (usa score_dec do melhor modelo)
sc1d_val  = best_svm_cnn.decision_function(GAP_val).astype(np.float64)
sc1d_test = best_svm_cnn.decision_function(GAP_test).astype(np.float64)
pd_cnn1d_svm, _ = d3f_pd_vs_snr(sc1d_val, sc1d_test, y_val, y_test, snr_val, snr_test, SNR_BINS)

# Tabela comparativa
print(f'\n{"SNR":>5}  {"Classical":>10}  {"CNN1D-SVM":>12}  {"CNN4-SVM":>12}')
print('-' * 46)
for s in SNR_BINS:
    cl  = classical_pd.get(s) or 0
    c1d = pd_cnn1d_svm.get(s) or 0
    c4  = pd_cnn4_svm.get(s) or 0
    print(f'{s:>5}  {cl:>10.4f}  {c1d:>12.4f}  {c4:>12.4f}')

# Gráfico final
snr_v_arr = np.array(SNR_BINS, dtype=float)
get = lambda d: np.array([d.get(s) or 0 for s in SNR_BINS])

fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle(f'CNN-SVM Híbrido: 1D z_eq vs 4-feat  (α={ALPHA:.0e}, D3F Gaussiano)',
             fontsize=12, fontweight='bold')

ax.plot(snr_v_arr, [classical_pd.get(s, 0) for s in SNR_BINS],
        'o-', color='tomato', lw=2.5, ms=8, label='Classical (Xie 2021)')
ax.plot(snr_v_arr, get(pd_cnn1d_svm),
        's--', color='steelblue', lw=2.5, ms=8,
        label=f'CNN 1D z_eq → SVM (128-d GAP)')
ax.plot(snr_v_arr, get(pd_cnn4_svm),
        'D-.', color='mediumpurple', lw=2.5, ms=8,
        label=f'CNN 4-feat → SVM (32-d GAP)')

ax.set(xlabel='SNR (dB)', ylabel='PD',
       xlim=(-1, 31), ylim=(-0.05, 1.05))
ax.set_xticks(SNR_BINS)
ax.legend(fontsize=10, loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = vis_dir / 'NN11_CNN_SVM_1d_vs_4feat.png'
plt.savefig(str(fig_path), dpi=150)
plt.show()
print(f'Figura salva → {fig_path}')

# Salvar resultados
import joblib
joblib.dump(best_svm4, str(models_dir / 'svm_cnn4feat_hybrid.pkl'))

results_ext = {
    'cnn4feat_svm': {
        'model': best_svm4_name, 'gap_dim': GAP_DIM_4,
        'pd_vs_snr': {str(k): pd_cnn4_svm.get(k) for k in SNR_BINS}
    },
    'cnn1d_svm': {
        'pd_vs_snr': {str(k): pd_cnn1d_svm.get(k) for k in SNR_BINS}
    }
}
out = data_dir / 'nn11_cnn_svm_comparison.json'
with open(str(out), 'w') as fp:
    json.dump(results_ext, fp, indent=2)
print(f'Resultados salvos → {out}')
